# Part 6: KV Cache Management

In Part 5, we built a continuous batching engine.

Our scheduler can now:

```text
admit requests
      ↓
prefill prompts
      ↓
decode active sequences
      ↓
remove finished requests
      ↓
admit new requests
      ↓
repeat
```

We also introduced token-budget-aware scheduling so the engine can reason about the amount of inference work performed during each iteration.

However, while building dynamic batches we repeatedly ran into another problem:

```text
KV-cache state
```

When requests entered or left the active batch, we sometimes had to:

```text
slice KV tensors
pad cache lengths
concatenate different caches
rebuild batched cache state
```

That worked for learning continuous batching, but it exposed an important limitation.

A serving engine cannot treat KV cache as just another tensor that gets casually rebuilt whenever the active batch changes.

The KV cache is one of the largest and most important consumers of GPU memory during LLM inference.

From this point onward, we need to treat it as a **managed resource**.

---

## Quick Recap - What Is the KV Cache?

During autoregressive generation, every transformer layer computes attention keys and values for the tokens it processes.

Without caching, generating a new token would require recomputing those K/V states for all previous tokens again.

Instead, after prefill:

```text
prompt tokens
     ↓
transformer
     ↓
K/V states
     ↓
KV CACHE
```

During decode, only the new token needs to be processed:

```text
new token
   +
existing KV cache
      ↓
transformer
      ↓
next token
```

This is why decode can reuse previous computation instead of repeatedly processing the entire sequence.

---

## The New Problem: Memory

Consider three requests:

```text
A → 200 cached tokens
B → 800 cached tokens
C → 3000 cached tokens
```

Each request needs KV memory for every cached token across every transformer layer.

As generation continues:

```text
A: 200 → 201 → 202 → ...

B: 800 → 801 → 802 → ...

C: 3000 → 3001 → 3002 → ...
```

Their KV-cache usage grows continuously.

Now imagine hundreds of concurrent requests doing this.

Eventually the question is no longer only:

```text
Which request should the scheduler run?
```

The engine must also ask:

```text
Do I actually have enough KV-cache memory
to keep this request running?
```

---

## KV Cache Has a Lifecycle

A request does not need KV memory forever.

Its cache follows roughly:

```text
request arrives
      ↓
KV space allocated
      ↓
prefill writes KV
      ↓
decode extends KV
      ↓
request finishes
      ↓
KV space released
```

So KV memory needs its own lifecycle:

```text
ALLOCATE
   ↓
USE
   ↓
GROW
   ↓
FREE
```

This means the inference engine needs something responsible for tracking:

```text
how much KV memory exists
how much is free
which request owns which memory
how much each request currently uses
when memory can be reclaimed
```

That component is the **KV-cache manager**.

---

## Why Our Dense Cache Approach Is Not Enough

In Part 5B, our cache looked conceptually like:

```text
KV tensor

[batch, kv_heads, sequence_length, head_dim]
```

For example:

```text
row 0 → Request B
row 1 → Request C
```

When requests changed, we manipulated the dense tensor:

```text
[A, B]
   ↓
remove A
   ↓
[B]

then

[B]
 ↓
merge C
 ↓
[B, C]
```

This required operations such as:

```text
select rows
pad sequence lengths
concatenate caches
rebuild cache objects
```

The underlying problem is that a dense tensor assumes the active requests fit neatly into one contiguous rectangular structure.

Real serving workloads do not behave neatly.

Requests have:

```text
different prompt lengths
different output lengths
different arrival times
different completion times
```

For example:

```text
Request A: ████████████

Request B: █████

Request C: █████████████████████

Request D: ██
```

Trying to continuously reshape all of this into one dense structure becomes increasingly awkward and expensive.

---

## Memory Cost of KV Cache

A simplified KV-cache memory estimate for one token is:

```text
2
× number_of_layers
× number_of_KV_heads
× head_dimension
× bytes_per_element
```

The `2` comes from storing both:

```text
Key
Value
```

So total KV memory grows approximately with:

```text
number of cached tokens
×
KV bytes per token
```

For one request:

```text
KV memory
≈
sequence length × bytes per token
```

For many requests:

```text
total KV memory
≈
sum of cached tokens across all active requests
×
bytes per token
```

This is why serving engines often reason about **total cached tokens**, not just the number of requests.

Two active requests could mean:

```text
Request A → 100 tokens
Request B → 100 tokens

total = 200 cached tokens
```

or:

```text
Request A → 20,000 tokens
Request B → 20,000 tokens

total = 40,000 cached tokens
```

Same number of requests.

Completely different GPU-memory requirement.

---

## Scheduler Capacity Is Not the Same as KV Capacity

In Part 5B, our scheduler used a limit such as:

```python
max_tokens_per_step = 50
```

That controls how much **compute work** we allow during one iteration.

But now we need another concept:

```text
KV capacity
```

For example:

```text
scheduler token budget
=
50 tokens this step
```

while:

```text
remaining KV capacity
=
only 10 cached tokens
```

A new 40-token request may fit the compute budget but still be impossible to admit because there is not enough memory to store its KV state.

So future scheduling decisions need to consider both:

```text
COMPUTE CAPACITY
+
MEMORY CAPACITY
```

---

## What We Will Build

In this part we will build a basic KV-cache manager that can reason about:

```text
total KV capacity
used capacity
free capacity
request ownership
allocation
growth
release
```

Conceptually:

```text
KV CACHE MANAGER

Total capacity: 100 tokens

Request A → 20
Request B → 35
Request C → 10

Used: 65

Free: 35
```

When Request A finishes:

```text
free(A)
```

the state becomes:

```text
Request B → 35
Request C → 10

Used: 45

Free: 55
```

That reclaimed capacity can then be reused by another request.

---

## Capacity-Aware Scheduling

Eventually our scheduler will need to ask the KV manager before admitting work.

Instead of:

```text
request fits token budget
        ↓
run request
```

we move toward:

```text
request fits compute budget?
        ↓
       yes
        ↓
enough KV capacity?
        ↓
       yes
        ↓
admit request
```

If KV memory is unavailable:

```text
request remains waiting
```

even if there is otherwise enough compute capacity.

---

## What This Part Will Not Solve Yet

Our first KV manager will intentionally be simple.

We will initially reason in terms of contiguous logical cache capacity.

That still leaves problems such as:

```text
fragmentation
expensive memory movement
variable-length allocations
growing sequences
```

Those problems lead directly to the next part.

---

## Where This Is Going

The progression is:

```text
Continuous batching
        ↓
many independently-growing sequences
        ↓
KV cache becomes a managed GPU resource
        ↓
KV-cache manager
        ↓
contiguous allocation problems
        ↓
paged KV allocation
        ↓
PagedAttention-style memory management
```

So Part 6 is not about making attention itself faster.

It is about answering a more fundamental serving question:

> **How does the inference engine safely allocate, track, grow, and reclaim the memory used by KV caches?**

Once that is clear, we will be ready to replace naive contiguous cache management with paged allocation.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
).to("cuda")

model.eval()

print("Model:", model_name)
print("Device:", model.device)
print("Dtype:", model.dtype)

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model: HuggingFaceTB/SmolLM2-1.7B-Instruct
Device: cuda:0
Dtype: torch.float16


In [3]:
# Inspect model architecture to estimate KV memory per token.

config = model.config

print("Layers:", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("KV heads:", config.num_key_value_heads)
print("Hidden size:", config.hidden_size)
print("Dtype:", model.dtype)

Layers: 24
Attention heads: 32
KV heads: 32
Hidden size: 2048
Dtype: torch.float16


In [4]:
# Calculate the attention head dimension.

head_dim = (
    config.hidden_size
    // config.num_attention_heads
)

print("Head dimension:", head_dim)

Head dimension: 64


In [5]:
bytes_per_element = torch.tensor(
    [],
    dtype=model.dtype
).element_size()

print("Bytes per element:", bytes_per_element)

Bytes per element: 2


In [6]:
kv_bytes_per_token = (
    2
    * config.num_hidden_layers
    * config.num_key_value_heads
    * head_dim
    * bytes_per_element
)

print("KV bytes per token:", kv_bytes_per_token)

print(
    "KV KB per token:",
    kv_bytes_per_token / 1024
)

KV bytes per token: 196608
KV KB per token: 192.0


In [7]:
# Simulate memory growth as context length increases.

context_lengths = [
    128,
    512,
    1024,
    4096,
    8192,
    16384,
]

for tokens in context_lengths:

    memory_mb = (
        tokens
        * kv_bytes_per_token
        / 1024**2
    )

    print(
        f"{tokens:>6} tokens"
        f" -> {memory_mb:.2f} MB"
    )

   128 tokens -> 24.00 MB
   512 tokens -> 96.00 MB
  1024 tokens -> 192.00 MB
  4096 tokens -> 768.00 MB
  8192 tokens -> 1536.00 MB
 16384 tokens -> 3072.00 MB


In [8]:
# Multiple concurrent requests.

cached_tokens = {
    "A": 4000,
    "B": 8000,
    "C": 2000,
}

total_cached_tokens = sum(
    cached_tokens.values()
)

total_kv_mb = (
    total_cached_tokens
    * kv_bytes_per_token
    / 1024**2
)

print(
    "Total cached tokens:",
    total_cached_tokens
)

print(
    f"Total KV memory: "
    f"{total_kv_mb:.2f} MB"
)

Total cached tokens: 14000
Total KV memory: 2625.00 MB


## Understanding KV-Cache Memory Cost

Before managing KV memory, we first need to know how much memory one cached token actually consumes.

For one token, every transformer layer stores both a **Key** and a **Value** for every KV head.

A simplified estimate is:

```text
KV bytes per token
=
2
× number of layers
× number of KV heads
× head dimension
× bytes per element
```

The `2` represents:

```text
Key + Value
```

For our current model, this gives:

```text
KV bytes per token: 196608
KV memory per token: 192 KB
```

This means KV memory grows linearly with the number of cached tokens.

Our measurements were:

```text
128 tokens    →   24 MB
512 tokens    →   96 MB
1,024 tokens  →  192 MB
4,096 tokens  →  768 MB
8,192 tokens  →  1.5 GB
16,384 tokens →  3 GB
```

So a long-running request can consume a significant amount of GPU memory even though the model weights themselves never change.

The situation becomes even more important with concurrent requests.

For example:

```text
Request A → 4,000 cached tokens
Request B → 8,000 cached tokens
Request C → 2,000 cached tokens
```

The server is holding:

```text
14,000 cached tokens
```

which for this model requires approximately:

```text
2.625 GB of KV-cache memory
```

This gives us an important serving insight:

> The number of active requests alone does not tell us how much KV memory is being used. What matters is the total number of cached tokens across those requests.

So the inference engine needs to track KV memory as a limited resource:

```text
total KV capacity
      ↓
allocated to requests
      ↓
grows during decoding
      ↓
released when requests finish
```

That is the job of the **KV-cache manager**, which we will build next.

In [17]:
# Create a basic KV-cache manager.


class KVCacheManager:

    def __init__(self, total_capacity):
        self.total_capacity = total_capacity
        self.allocations = {}

    @property
    def used_capacity(self):
        return sum(self.allocations.values())

    @property
    def free_capacity(self):
        return self.total_capacity - self.used_capacity

    def allocate(self, request_id, num_tokens):

        if num_tokens > self.free_capacity:
            return False

        self.allocations[request_id] = num_tokens

        return True

In [18]:
# Dry run.

kv_manager = KVCacheManager(
    total_capacity=10_000
)

print(kv_manager.allocate("A", 4000))
print(kv_manager.allocate("B", 3000))

print(kv_manager.allocations)

print("Used:", kv_manager.used_capacity)
print("Free:", kv_manager.free_capacity)


True
True
{'A': 4000, 'B': 3000}
Used: 7000
Free: 3000


In [19]:
# Try to fit a request above capacity.

print(
    kv_manager.allocate(
        "C",
        4000
    )
)


False


In [ ]:
# KV cache grows during decode.

def grow(self, request_id, num_tokens=1):

    if request_id not in self.allocations:
        # The request must already own KV capacity.
        raise ValueError(
            f"{request_id} has no KV allocation"
        )
    if num_tokens > self.free_capacity:
        return False

    self.allocations[request_id] += num_tokens

    return True

In [20]:
# The full class becomes:

class KVCacheManager:

    def __init__(self, total_capacity):
        self.total_capacity = total_capacity
        self.allocations = {}

    @property
    def used_capacity(self):
        return sum(self.allocations.values())

    @property
    def free_capacity(self):
        return self.total_capacity - self.used_capacity

    def allocate(self, request_id, num_tokens):

        if num_tokens > self.free_capacity:
            return False

        self.allocations[request_id] = num_tokens

        return True

    def grow(self, request_id, num_tokens=1):

        if request_id not in self.allocations:
            raise ValueError(
                f"{request_id} has no KV allocation"
            )

        if num_tokens > self.free_capacity:
            return False

        self.allocations[request_id] += num_tokens

        return True



In [21]:
#dry run

kv_manager = KVCacheManager(10_000)

kv_manager.allocate("A", 4000)
kv_manager.allocate("B", 3000)

# Decode iteration 1.
kv_manager.grow("A")

print(kv_manager.allocations)



{'A': 4001, 'B': 3000}


In [ ]:
# Function to free a request's KV cache.

def free(self, request_id):

    if request_id not in self.allocations:
        return

    del self.allocations[request_id]

In [22]:
# Add that method to the full manager class.

class KVCacheManager:

    def __init__(self, total_capacity):
        self.total_capacity = total_capacity
        self.allocations = {}

    @property
    def used_capacity(self):
        return sum(self.allocations.values())

    @property
    def free_capacity(self):
        return self.total_capacity - self.used_capacity

    def allocate(self, request_id, num_tokens):

        if num_tokens > self.free_capacity:
            return False

        self.allocations[request_id] = num_tokens

        return True

    def grow(self, request_id, num_tokens=1):

        if request_id not in self.allocations:
            raise ValueError(
                f"{request_id} has no KV allocation"
            )

        if num_tokens > self.free_capacity:
            return False

        self.allocations[request_id] += num_tokens

        return True

    def free(self, request_id):

        if request_id not in self.allocations:
            return

        del self.allocations[request_id]

In [23]:
#dry run


kv_manager = KVCacheManager(10_000)

kv_manager.allocate("A", 4000)
kv_manager.allocate("B", 3000)

kv_manager.grow("A")
kv_manager.grow("A")

print("Before A finishes:")
print(kv_manager.allocations)
print("Free:", kv_manager.free_capacity)

Before A finishes:
{'A': 4002, 'B': 3000}
Free: 2998


In [24]:
kv_manager.free("A")

print("\nAfter A finishes:")
print(kv_manager.allocations)
print("Free:", kv_manager.free_capacity)


After A finishes:
{'B': 3000}
Free: 7000


## Building a Basic KV-Cache Manager

Now that we know KV memory scales with the number of cached tokens, we need a component that tracks that memory as a limited resource.

Our first KV-cache manager tracks:

```text
total capacity
used capacity
free capacity
request allocations
```

Conceptually:

```text
KV capacity = 10,000 token slots

A → 4,000
B → 3,000

Used = 7,000
Free = 3,000
```

### Allocation

When a request is admitted, it reserves KV capacity for its current tokens:

```python
kv_manager.allocate("A", 4000)
```

If enough capacity exists, allocation succeeds.

If not:

```python
kv_manager.allocate("C", 4000)
```

returns `False`, and C cannot be admitted yet.

This gives the scheduler a new constraint beyond compute capacity:

```text
request fits compute budget
        +
request fits KV capacity
        ↓
request can run
```

### Growth During Decode

KV usage increases as generation continues.

If A has:

```text
4000 cached tokens
```

then after one decode step:

```text
4001
```

and after another:

```text
4002
```

We model this with:

```python
kv_manager.grow("A")
```

So KV allocation follows the actual autoregressive sequence length.

### Freeing Memory

When a request finishes, its cached K/V state is no longer needed.

```python
kv_manager.free("A")
```

releases A's allocation so the capacity can be reused by another request.

The complete lifecycle is:

```text
request arrives
      ↓
allocate KV
      ↓
prefill
      ↓
decode grows KV
      ↓
request finishes
      ↓
free KV
```

At this point our manager tracks **logical KV capacity in token slots** rather than allocating real K/V tensors itself.

That is enough to introduce the next serving behavior: **capacity-aware scheduling**, where a request may remain waiting even when compute capacity exists because there is not enough KV memory available.

In [25]:

# PART 6: CAPACITY-AWARE SCHEDULING

# Scheduler now checks 2 resources:
# 1. Compute capacity (max_tokens_per_step)
# 2. KV-cache capacity (kv_manager.free_capacity_


class KVCacheManager:

    def __init__(self, total_capacity):
        # Capacity is measured in cached-token slots
        self.total_capacity = total_capacity

        # request_id -> cached token count
        self.allocations = {}


    @property
    def used_capacity(self):
        return sum(self.allocations.values())


    @property
    def free_capacity(self):
        return (
            self.total_capacity
            - self.used_capacity
        )


    def can_allocate(self, num_tokens):
        return num_tokens <= self.free_capacity


    def allocate(self, request_id, num_tokens):

        if not self.can_allocate(num_tokens):
            return False

        self.allocations[request_id] = num_tokens

        return True


    def grow(self, request_id, num_tokens=1):

        if request_id not in self.allocations:
            raise ValueError(
                f"{request_id} has no KV allocation"
            )

        if num_tokens > self.free_capacity:
            return False

        self.allocations[request_id] += num_tokens

        return True


    def free(self, request_id):

        if request_id in self.allocations:
            del self.allocations[request_id]

# REQUEST

def create_request(
    request_id,
    prompt_tokens,
    max_new_tokens
):

    return {
        "id": request_id,

        # Number of tokens that must be cached during prefill
        "prompt_tokens": prompt_tokens,

        "generated_tokens": 0,

        "max_new_tokens": max_new_tokens,

        "finished": False,
    }

# ENGINE CONFIG

max_running = 2

# Maximum token work per engine iteration
max_tokens_per_step = 8

# Total KV cache can hold 12 token positions
kv_manager = KVCacheManager(
    total_capacity=12
)

# REQUEST QUEUES

waiting = [
    create_request(
        "A",
        prompt_tokens=5,
        max_new_tokens=3,
    ),

    create_request(
        "B",
        prompt_tokens=5,
        max_new_tokens=3,
    ),

    create_request(
        "C",
        prompt_tokens=4,
        max_new_tokens=2,
    ),
]

running = []

finished = []


# ENGINE LOOP


step = 0
max_steps = 20


while waiting or running:

    step += 1

    if step > max_steps:
        break


    remaining_compute = (
        max_tokens_per_step
    )

    decoded = []
    admitted = []
    finished_this_step = []
    kv_blocked = []

    made_progress = False


    # 1. DECODE EXISTING REQUESTS

    # A decode step costs:
    # compute: 1 token
    # KV memory: 1 additional token slot


    for request in running[:]:

        if remaining_compute < 1:
            break

        # Before decoding, we must be able to grow
        # this request's KV cache by one token.

        success = kv_manager.grow(
            request["id"],
            num_tokens=1,
        )

        if not success:

            kv_blocked.append(
                request["id"]
            )

            continue

        # Simulate generation of one new token

        request["generated_tokens"] += 1

        remaining_compute -= 1

        decoded.append(
            request["id"]
        )

        made_progress = True


        # Check whether request finished

        if (
            request["generated_tokens"]
            >= request["max_new_tokens"]
        ):

            request["finished"] = True

            running.remove(request)

            finished.append(request)

            finished_this_step.append(
                request["id"]
            )


            # Entire KV allocation can now be reclaimed.

            kv_manager.free(
                request["id"]
            )


    # 2. ADMIT WAITING REQUESTS


    # Prefill requires:

    # compute = prompt_tokens
    # KV    = prompt_tokens

    while (
        waiting
        and len(running) < max_running
    ):

        request = waiting[0]

        prefill_cost = (
            request["prompt_tokens"]
        )



        # CHECK COMPUTE CAPACITY


        if prefill_cost > remaining_compute:

            break


        # CHECK KV CAPACITY


        if not kv_manager.can_allocate(
            prefill_cost
        ):

            break


        # Both resources are available.
        # The request may now be admitted.

        waiting.pop(0)


        kv_manager.allocate(
            request["id"],
            prefill_cost,
        )


        running.append(
            request
        )


        remaining_compute -= (
            prefill_cost
        )


        admitted.append(
            request["id"]
        )

        made_progress = True



    # 3. TRACE ENGINE STATE


    print(f"\nSTEP {step}")

    print(
        "  decode:",
        decoded
    )

    print(
        "  prefill/admit:",
        admitted
    )

    print(
        "  finished:",
        finished_this_step
    )

    print(
        "  KV blocked:",
        kv_blocked
    )

    print(
        "  running:",
        [
            request["id"]
            for request in running
        ]
    )

    print(
        "  waiting:",
        [
            request["id"]
            for request in waiting
        ]
    )


    print(
        "  compute used:",
        max_tokens_per_step
        - remaining_compute,
        "/",
        max_tokens_per_step,
    )


    print(
        "  KV:",
        kv_manager.used_capacity,
        "/",
        kv_manager.total_capacity,
        "used"
    )


    print(
        "  KV allocations:",
        kv_manager.allocations
    )



    # 4. DETECT KV DEADLOCK / MEMORY EXHAUSTION


    if not made_progress:

        print(
            "\nENGINE STALLED:"
        )

        print(
            "No request could make progress."
        )

        print(
            "Available KV slots:",
            kv_manager.free_capacity
        )

        break


# FINAL STATE


print(
    "\n\n========== FINAL STATE =========="
)

print(
    "Finished:",
    [
        request["id"]
        for request in finished
    ]
)

print(
    "Running:",
    [
        request["id"]
        for request in running
    ]
)

print(
    "Waiting:",
    [
        request["id"]
        for request in waiting
    ]
)

print(
    "KV allocations:",
    kv_manager.allocations
)

print(
    "Free KV capacity:",
    kv_manager.free_capacity
)


STEP 1
  decode: []
  prefill/admit: ['A']
  finished: []
  KV blocked: []
  running: ['A']
  waiting: ['B', 'C']
  compute used: 5 / 8
  KV: 5 / 12 used
  KV allocations: {'A': 5}

STEP 2
  decode: ['A']
  prefill/admit: ['B']
  finished: []
  KV blocked: []
  running: ['A', 'B']
  waiting: ['C']
  compute used: 6 / 8
  KV: 11 / 12 used
  KV allocations: {'A': 6, 'B': 5}

STEP 3
  decode: ['A']
  prefill/admit: []
  finished: []
  KV blocked: ['B']
  running: ['A', 'B']
  waiting: ['C']
  compute used: 1 / 8
  KV: 12 / 12 used
  KV allocations: {'A': 7, 'B': 5}

STEP 4
  decode: []
  prefill/admit: []
  finished: []
  KV blocked: ['A', 'B']
  running: ['A', 'B']
  waiting: ['C']
  compute used: 0 / 8
  KV: 12 / 12 used
  KV allocations: {'A': 7, 'B': 5}

ENGINE STALLED:
No request could make progress.
Available KV slots: 0


========== FINAL STATE ==========
Finished: []
Running: ['A', 'B']
Waiting: ['C']
KV allocations: {'A': 7, 'B': 5}
Free KV capacity: 0


## Capacity-Aware Scheduling

Our scheduler previously considered mainly **compute capacity**.

For example:

```python
max_tokens_per_step = 8
```

This limits how much token work can happen during one engine iteration.

However, a request also needs enough GPU memory to store its KV cache.

So the scheduler now has to consider two separate resources:

```text
Compute capacity
+
KV-cache capacity
```

A request should only be admitted if both are available.

---

### Scheduler Checks

For a new request, prefill requires:

```text
compute cost = prompt tokens
KV cost      = prompt tokens
```

For an existing decoding request:

```text
compute cost = 1 token
KV growth    = 1 token slot
```

So before running work, the scheduler checks:

```text
Does it fit the compute budget?
        ↓
Does it fit the KV capacity?
        ↓
       run
```

If KV capacity is unavailable, the request cannot make progress even if plenty of compute budget remains.

---

### Initial Configuration

Our experiment used:

```text
max_running = 2

compute budget per step = 8 tokens

KV capacity = 12 cached-token slots
```

The requests were:

```text
A
prompt = 5 tokens
max_new_tokens = 3

B
prompt = 5 tokens
max_new_tokens = 3

C
prompt = 4 tokens
max_new_tokens = 2
```

---

### Step 1

The engine produced:

```text
STEP 1
  decode: []
  prefill/admit: ['A']
  running: ['A']
  waiting: ['B', 'C']
  compute used: 5 / 8
  KV: 5 / 12 used
```

A requires five prompt tokens.

Its prefill therefore consumes:

```text
5 compute tokens
5 KV slots
```

After admitting A:

```text
KV used = 5
KV free = 7
```

B remains waiting because only three compute tokens remain in this iteration:

```text
8 - 5 = 3
```

but B's prefill requires five.

So B fits in KV memory, but does not fit the remaining compute budget.

---

### Step 2

```text
STEP 2
  decode: ['A']
  prefill/admit: ['B']
  running: ['A', 'B']
  waiting: ['C']
  compute used: 6 / 8
  KV: 11 / 12 used
```

A performs one decode step.

That requires:

```text
1 compute token
1 new KV slot
```

A's KV allocation grows:

```text
5 → 6
```

B can then be prefetched:

```text
B prefill = 5 tokens
```

The total work this iteration is:

```text
A decode  = 1
B prefill = 5
----------------
total     = 6
```

KV usage becomes:

```text
A = 6
B = 5
------
   11 / 12
```

Only one KV slot remains.

---

### Step 3

```text
STEP 3
  decode: ['A']
  prefill/admit: []
  KV blocked: ['B']
  running: ['A', 'B']
  waiting: ['C']
  compute used: 1 / 8
  KV: 12 / 12 used
```

A gets processed first.

It needs one additional KV slot:

```text
A = 6 → 7
```

That consumes the final free KV slot.

The cache is now completely full:

```text
A = 7
B = 5
------
   12 / 12
```

The scheduler then reaches B.

B also wants to generate one token.

Its compute requirement is only:

```text
1 token
```

and plenty of compute budget remains.

But decoding B would require its KV allocation to grow:

```text
B = 5 → 6
```

There are no free KV slots.

So:

```text
KV blocked: ['B']
```

This demonstrates an important distinction:

```text
available compute
does not imply
available serving capacity
```

The GPU can theoretically perform more compute, but the engine has nowhere to store the new K/V state.

---

### Step 4

```text
STEP 4
  decode: []
  prefill/admit: []
  KV blocked: ['A', 'B']
  running: ['A', 'B']
  waiting: ['C']
  compute used: 0 / 8
  KV: 12 / 12 used
```

Now the problem becomes worse.

Both A and B need another KV slot to generate their next token.

But:

```text
free KV capacity = 0
```

So neither request can decode.

C also cannot be admitted because there is no free KV capacity.

The engine reaches:

```text
ENGINE STALLED
```

with:

```text
Running: ['A', 'B']
Waiting: ['C']

KV allocations:
A = 7
B = 5

Free KV capacity = 0
```

---

### Why This Is Important

This experiment exposes a failure mode that our earlier scheduler could not see.

Previously, we might have looked only at:

```text
compute budget = 8

decode A = 1
decode B = 1
```

and concluded:

```text
1 + 1 <= 8

Both requests can run.
```

But that is incomplete.

Each generated token also needs persistent KV memory.

The real decision is closer to:

```text
Can I afford the compute?
        AND
Can I store the resulting KV state?
```

In this example:

```text
Compute available: YES

KV memory available: NO

Result: cannot decode
```

---

### KV Memory Can Block Existing Requests Too

KV exhaustion does not only prevent new requests from entering.

It can stop requests that are **already running**.

Every decode step extends the sequence:

```text
100 cached tokens
      ↓
generate token
      ↓
101 cached tokens
```

So the KV cache must continue growing throughout generation.

If no space remains:

```text
decode
  ↓
needs +1 KV slot
  ↓
no free capacity
  ↓
request blocked
```

Our simulation reached exactly this state.

---

### The Limitation of Our Current Manager

Our manager currently thinks about KV memory as one simple pool:

```text
12 token slots total

A owns 7
B owns 5

free = 0
```

It understands:

```text
allocate
grow
free
capacity checks
```

but it does not yet understand **where those token slots physically live in GPU memory**.

It also has no mechanism for:

```text
preemption
moving requests
fragmentation
non-contiguous allocation
reusing small pieces of freed memory efficiently
```

This means we now understand the scheduling side of KV-cache management, but not yet the physical memory-layout problem.

That is the final limitation we need to examine before moving to paged KV allocation.

In [26]:
# Implement a KV-aware scheduler with real inference.


import torch
from transformers import DynamicCache


# Tracks logical KV capacity in cached-token slots.
class KVCacheManager:
    def __init__(self, total_capacity):
        self.total_capacity = total_capacity

        # request_id -> number of cached tokens
        self.allocations = {}

    @property
    def used_capacity(self):
        return sum(self.allocations.values())

    @property
    def free_capacity(self):
        return self.total_capacity - self.used_capacity

    def can_allocate(self, num_tokens):
        return num_tokens <= self.free_capacity

    def allocate(self, request_id, num_tokens):
        if not self.can_allocate(num_tokens):
            return False

        self.allocations[request_id] = num_tokens
        return True

    def grow(self, request_id, num_tokens=1):
        if request_id not in self.allocations:
            raise ValueError(
                f"{request_id} has no KV allocation"
            )

        if num_tokens > self.free_capacity:
            return False

        self.allocations[request_id] += num_tokens
        return True

    def free(self, request_id):
        self.allocations.pop(request_id, None)


# Create a request using the instruct model's chat template.
def create_request(
    request_id,
    prompt,
    max_new_tokens,
):
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    input_ids = inputs["input_ids"].to(
        model.device
    )

    attention_mask = inputs["attention_mask"].to(
        model.device
    )

    return {
        "id": request_id,
        "prompt": prompt,

        "input_ids": input_ids,
        "attention_mask": attention_mask,

        # Actual prompt size after tokenization.
        "prompt_tokens": input_ids.shape[-1],

        # Each request owns a real DynamicCache.
        "cache": None,

        # Token that will be processed during the next decode step.
        "next_token": None,

        "generated_tokens": [],

        "max_new_tokens": max_new_tokens,

        "finished": False,
    }


# Process the complete prompt and create its KV cache.
@torch.inference_mode()
def run_prefill(request):
    cache = DynamicCache(
        config=model.config
    )

    prompt_length = (
        request["input_ids"].shape[-1]
    )

    cache_position = torch.arange(
        prompt_length,
        device=model.device,
    )

    outputs = model(
        input_ids=request["input_ids"],
        attention_mask=request["attention_mask"],
        past_key_values=cache,
        cache_position=cache_position,
        use_cache=True,
    )

    request["cache"] = (
        outputs.past_key_values
    )

    next_token = (
        outputs.logits[:, -1, :]
        .argmax(
            dim=-1,
            keepdim=True,
        )
    )

    request["next_token"] = next_token

    request["generated_tokens"].append(
        next_token.item()
    )


# Process one real decode token using the existing KV cache.
@torch.inference_mode()
def run_decode(request):
    past_length = (
        request["cache"].get_seq_length()
    )

    # The visible context becomes one token longer.
    request["attention_mask"] = torch.cat(
        [
            request["attention_mask"],

            request["attention_mask"].new_ones(
                (1, 1)
            ),
        ],
        dim=-1,
    )

    cache_position = torch.tensor(
        [past_length],
        dtype=torch.long,
        device=model.device,
    )

    outputs = model(
        input_ids=request["next_token"],
        attention_mask=request["attention_mask"],
        past_key_values=request["cache"],
        cache_position=cache_position,
        use_cache=True,
    )

    request["cache"] = (
        outputs.past_key_values
    )

    next_token = (
        outputs.logits[:, -1, :]
        .argmax(
            dim=-1,
            keepdim=True,
        )
    )

    request["next_token"] = next_token

    request["generated_tokens"].append(
        next_token.item()
    )


# Stop on max_new_tokens or EOS.
def request_finished(request):
    reached_limit = (
        len(request["generated_tokens"])
        >= request["max_new_tokens"]
    )

    generated_eos = (
        len(request["generated_tokens"]) > 0
        and
        request["generated_tokens"][-1]
        == tokenizer.eos_token_id
    )

    return reached_limit or generated_eos


# Make sure our logical KV accounting matches the real cache.
def verify_kv_accounting(
    request,
    kv_manager,
):
    actual_cache_tokens = (
        request["cache"].get_seq_length()
    )

    tracked_cache_tokens = (
        kv_manager.allocations[
            request["id"]
        ]
    )

    assert (
        actual_cache_tokens
        == tracked_cache_tokens
    ), (
        f"KV accounting mismatch for "
        f"{request['id']}: "
        f"actual={actual_cache_tokens}, "
        f"tracked={tracked_cache_tokens}"
    )


class CapacityAwareScheduler:
    def __init__(
        self,
        requests,
        kv_manager,
        max_running,
        max_tokens_per_step,
    ):
        self.waiting = list(requests)
        self.running = []
        self.finished = []

        self.kv_manager = kv_manager

        self.max_running = max_running
        self.max_tokens_per_step = (
            max_tokens_per_step
        )

    # Decide which running requests are allowed to decode.
    def plan_decodes(self, compute_budget):
        scheduled = []
        kv_blocked = []

        projected_free_kv = (
            self.kv_manager.free_capacity
        )

        remaining_compute = compute_budget

        for request in self.running:
            # One decode iteration processes one token.
            if remaining_compute < 1:
                break

            # Processing that token also extends KV by one slot.
            if projected_free_kv < 1:
                kv_blocked.append(
                    request["id"]
                )
                continue

            scheduled.append(request)

            remaining_compute -= 1
            projected_free_kv -= 1

        return (
            scheduled,
            kv_blocked,
            remaining_compute,
        )

    # Decide which waiting requests can perform prefill.
    def plan_prefills(self, compute_budget):
        scheduled = []
        blocked_reason = None

        remaining_compute = compute_budget

        projected_free_kv = (
            self.kv_manager.free_capacity
        )

        projected_running = len(
            self.running
        )

        for request in self.waiting:
            if (
                projected_running
                >= self.max_running
            ):
                break

            prefill_cost = (
                request["prompt_tokens"]
            )

            # No chunked prefill yet.
            # The entire prompt must fit the compute budget.
            if prefill_cost > remaining_compute:
                blocked_reason = (
                    f"{request['id']} blocked by "
                    f"compute budget"
                )
                break

            # The complete prompt also needs KV capacity.
            if prefill_cost > projected_free_kv:
                blocked_reason = (
                    f"{request['id']} blocked by "
                    f"KV capacity"
                )
                break

            scheduled.append(request)

            remaining_compute -= prefill_cost
            projected_free_kv -= prefill_cost
            projected_running += 1

        return (
            scheduled,
            blocked_reason,
            remaining_compute,
        )

    def admit(self, request):
        self.waiting.remove(request)
        self.running.append(request)

    # Remove completed requests and reclaim their KV memory.
    def reap_finished(self):
        completed = []

        for request in self.running[:]:
            if not request_finished(request):
                continue

            request["finished"] = True

            self.running.remove(request)
            self.finished.append(request)

            completed.append(
                request["id"]
            )

            self.kv_manager.free(
                request["id"]
            )

            # Drop references to actual GPU KV tensors.
            request["cache"] = None
            request["next_token"] = None

        return completed


# Create real requests.
requests = [
    create_request(
        "A",
        "What is the capital of France? Answer briefly.",
        max_new_tokens=3,
    ),

    create_request(
        "B",
        "What is the largest planet in our solar system? Answer briefly.",
        max_new_tokens=6,
    ),

    create_request(
        "C",
        "What is machine learning? Answer briefly.",
        max_new_tokens=4,
    ),
]


print("Actual prompt lengths:")

for request in requests:
    print(
        request["id"],
        "->",
        request["prompt_tokens"],
        "tokens"
    )


# Use a deliberately tight KV budget.
# A and B can fit, but C cannot immediately join them.
kv_capacity = (
    requests[0]["prompt_tokens"]
    + requests[1]["prompt_tokens"]
    + 5
)

kv_manager = KVCacheManager(
    total_capacity=kv_capacity
)


scheduler = CapacityAwareScheduler(
    requests=requests,
    kv_manager=kv_manager,

    # Scheduler slots are not the bottleneck here.
    # KV memory should become the limiting resource.
    max_running=3,

    max_tokens_per_step=100,
)


print(
    "\nKV capacity:",
    kv_manager.total_capacity,
    "token slots"
)


step = 0
max_steps = 30


while (
    scheduler.waiting
    or scheduler.running
):
    step += 1

    if step > max_steps:
        raise RuntimeError(
            "Exceeded max engine steps"
        )

    compute_remaining = (
        scheduler.max_tokens_per_step
    )

    decoded = []
    prefetched = []
    kv_blocked = []
    finished_this_step = []

    prefill_block_reason = None


    # Scheduler chooses which active requests may decode.
    (
        decode_plan,
        kv_blocked,
        compute_remaining,
    ) = scheduler.plan_decodes(
        compute_remaining
    )


    # Execute the scheduled decode work.
    for request in decode_plan:
        # Reserve KV space before running the model.
        success = kv_manager.grow(
            request["id"],
            1,
        )

        assert success

        # Real model decode.
        run_decode(request)

        # Check that our logical manager matches the real cache.
        verify_kv_accounting(
            request,
            kv_manager,
        )

        decoded.append(
            request["id"]
        )


    # Finished requests free KV before new prefills are considered.
    finished_this_step.extend(
        scheduler.reap_finished()
    )


    # Scheduler chooses which new requests may prefill.
    (
        prefill_plan,
        prefill_block_reason,
        compute_remaining,
    ) = scheduler.plan_prefills(
        compute_remaining
    )


    # Execute real prompt prefills.
    for request in prefill_plan:
        prompt_tokens = (
            request["prompt_tokens"]
        )

        # Reserve KV capacity before model execution.
        success = kv_manager.allocate(
            request["id"],
            prompt_tokens,
        )

        assert success

        scheduler.admit(request)

        try:
            run_prefill(request)

        except Exception:
            # Roll back if model execution fails.
            kv_manager.free(
                request["id"]
            )

            scheduler.running.remove(
                request
            )

            scheduler.waiting.insert(
                0,
                request
            )

            raise

        verify_kv_accounting(
            request,
            kv_manager,
        )

        prefetched.append(
            request["id"]
        )


    # Prefill itself may generate EOS.
    finished_this_step.extend(
        scheduler.reap_finished()
    )


    print(f"\nSTEP {step}")

    print(
        "  decoded:",
        decoded
    )

    print(
        "  prefetched:",
        prefetched
    )

    print(
        "  finished:",
        finished_this_step
    )

    print(
        "  KV-blocked decode:",
        kv_blocked
    )

    if prefill_block_reason:
        print(
            "  prefill blocked:",
            prefill_block_reason
        )

    print(
        "  running:",
        [
            request["id"]
            for request in scheduler.running
        ]
    )

    print(
        "  waiting:",
        [
            request["id"]
            for request in scheduler.waiting
        ]
    )

    print(
        "  KV allocations:",
        kv_manager.allocations
    )

    print(
        "  KV used:",
        kv_manager.used_capacity,
        "/",
        kv_manager.total_capacity
    )


    # Actual DynamicCache lengths should match our bookkeeping.
    for request in scheduler.running:
        print(
            f"    {request['id']} actual cache:",
            request["cache"].get_seq_length(),
        )


    made_progress = (
        len(decoded) > 0
        or
        len(prefetched) > 0
        or
        len(finished_this_step) > 0
    )

    if not made_progress:
        print("\nENGINE STALLED")

        print(
            "Free KV slots:",
            kv_manager.free_capacity
        )

        break


print(
    "\nGENERATED OUTPUTS"
)

for request in scheduler.finished:
    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True,
    )

    print(
        f"\nRequest {request['id']}:"
    )

    print(text)


print(
    "\nFinal KV allocations:",
    kv_manager.allocations
)

print(
    "Final free KV slots:",
    kv_manager.free_capacity
)




Actual prompt lengths:
A -> 40 tokens
B -> 43 tokens
C -> 38 tokens

KV capacity: 88 token slots

STEP 1
  decoded: []
  prefetched: ['A', 'B']
  finished: []
  KV-blocked decode: []
  prefill blocked: C blocked by compute budget
  running: ['A', 'B']
  waiting: ['C']
  KV allocations: {'A': 40, 'B': 43}
  KV used: 83 / 88
    A actual cache: 40
    B actual cache: 43

STEP 2
  decoded: ['A', 'B']
  prefetched: []
  finished: []
  KV-blocked decode: []
  prefill blocked: C blocked by KV capacity
  running: ['A', 'B']
  waiting: ['C']
  KV allocations: {'A': 41, 'B': 44}
  KV used: 85 / 88
    A actual cache: 41
    B actual cache: 44

STEP 3
  decoded: ['A', 'B']
  prefetched: ['C']
  finished: ['A']
  KV-blocked decode: []
  running: ['B', 'C']
  waiting: []
  KV allocations: {'B': 45, 'C': 38}
  KV used: 83 / 88
    B actual cache: 45
    C actual cache: 38

STEP 4
  decoded: ['B', 'C']
  prefetched: []
  finished: []
  KV-blocked decode: []
  running: ['B', 'C']
  waiting: []
  KV a

## Capacity-Aware Scheduling with Real Model Inference

So far, our KV-cache manager could track:

```text
allocate
grow
free
```

but our first scheduler experiment only simulated generation.

We now connect KV-capacity decisions to **actual model execution**.

Each request now owns:

```text
real tokenized prompt
real attention mask
real DynamicCache
real generated tokens
```

and the scheduler decides whether the model is allowed to perform:

```text
PREFILL
```

or:

```text
DECODE
```

based on two resources:

```text
compute budget
+
KV-cache capacity
```

This gives us a more realistic serving loop.

---

### Initial Request Sizes

After applying the model's chat template, our requests contained:

```text
A -> 40 tokens
B -> 43 tokens
C -> 38 tokens
```

Our KV manager was configured with:

```text
KV capacity = 88 cached-token slots
```

The scheduler also had:

```text
max_tokens_per_step = 100
```

So compute capacity is fairly generous.

KV memory is intentionally much tighter.

This lets us observe situations where:

```text
compute capacity exists
```

but:

```text
KV memory prevents execution
```

---

### Real KV State

Each active request owns an actual Hugging Face `DynamicCache`.

For example:

```text
Request A

logical allocation:
40 tokens

actual DynamicCache:
40 cached positions
```

After every forward pass, we verify:

```text
tracked KV tokens
==
actual DynamicCache sequence length
```

If these ever disagree, the program raises an error.

This is important because the KV manager is no longer tracking imaginary state.

Its bookkeeping corresponds directly to the cache created by real transformer forwards.

---

### Step 1

Our first scheduler iteration produced:

```text
STEP 1

decoded: []

prefetched:
['A', 'B']

finished:
[]

prefill blocked:
C blocked by compute budget

running:
['A', 'B']

waiting:
['C']

KV allocations:
A = 40
B = 43

KV used:
83 / 88
```

Initially there are no running requests.

So there is nothing to decode.

The scheduler starts considering prefills.

---

### Prefill A

A requires:

```text
40 prompt tokens
```

Prefill therefore costs:

```text
compute = 40 tokens
KV      = 40 slots
```

Both resources are available.

So A is admitted and a real model forward processes its complete prompt.

The result is:

```text
A DynamicCache length = 40
```

and:

```text
KV manager allocation = 40
```

---

### Prefill B

There are still:

```text
100 - 40
=
60 compute tokens
```

available.

B needs:

```text
43
```

so B also fits.

Its complete prompt is prefetched and creates:

```text
B DynamicCache length = 43
```

Now total KV usage is:

```text
A = 40
B = 43
------
   83
```

leaving only:

```text
88 - 83
=
5 KV slots
```

---

### Why C Does Not Run

After processing A and B, the compute budget contains:

```text
100 - 40 - 43
=
17
```

tokens.

But C requires:

```text
38 prompt tokens
```

for its prefill.

Since:

```text
38 > 17
```

C cannot fit into this iteration's remaining compute budget.

So C stays:

```text
waiting
```

Notice that in this particular step C fails the **compute check first**.

---

### Step 2

The next iteration gives us:

```text
STEP 2

decoded:
['A', 'B']

prefetched:
[]

prefill blocked:
C blocked by KV capacity

KV allocations:
A = 41
B = 44

KV used:
85 / 88
```

Now A and B have already completed prefill.

They are in decode mode.

---

### Real Decode Cost

During decode, we no longer process the complete prompt.

Each request sends only:

```text
latest generated token
+
existing KV cache
```

So each decode request consumes:

```text
compute = 1 token
KV growth = 1 token slot
```

A grows from:

```text
40 -> 41
```

and B grows from:

```text
43 -> 44
```

The actual caches confirm this:

```text
A actual cache: 41
B actual cache: 44
```

Total KV usage becomes:

```text
41 + 44
=
85 / 88
```

Only three slots remain.

---

### Why C Is Now Blocked by KV Memory

C still requires:

```text
38 KV slots
```

for its initial prompt.

Compute capacity is no longer the problem.

The scheduler still has plenty of token budget.

But:

```text
free KV = 3
```

while:

```text
C requires = 38
```

So the scheduler reports:

```text
C blocked by KV capacity
```

This is the first important serving behavior.

A request can have enough compute budget available and still remain waiting because there is nowhere to store its KV cache.

---

### Step 3

Step 3 is the most important transition:

```text
STEP 3

decoded:
['A', 'B']

prefetched:
['C']

finished:
['A']

running:
['B', 'C']

waiting:
[]

KV allocations:
B = 45
C = 38

KV used:
83 / 88
```

First, A and B perform another real decode step.

Their caches temporarily grow:

```text
A:
41 -> 42

B:
44 -> 45
```

At this point A reaches its generation stopping condition.

---

### A Finishes

When A finishes, its cache is no longer needed.

So the engine performs:

```text
free(A)
```

and also drops the reference to A's real `DynamicCache`.

Before freeing A:

```text
A = 42
B = 45

total = 87
```

After freeing A:

```text
B = 45

free capacity =
88 - 45
=
43
```

Suddenly there is enough space for C.

C requires:

```text
38 KV slots
```

so the scheduler can now admit it during the **same engine iteration**.

---

### Prefill C

C performs a real prompt forward pass.

Its cache becomes:

```text
C actual cache = 38
```

The KV manager records:

```text
C allocation = 38
```

Now:

```text
B = 45
C = 38
------
   83 / 88
```

The active set has changed:

```text
[A, B]
   ↓
A finishes
   ↓
[B]
   ↓
C admitted
   ↓
[B, C]
```

This connects our KV manager directly to continuous batching.

A finished request frees memory, and a waiting request immediately reuses that capacity.

---

### Step 4

Now both B and C are decoding:

```text
STEP 4

decoded:
['B', 'C']

running:
['B', 'C']

KV allocations:
B = 46
C = 39

KV used:
85 / 88
```

Both real caches grow by one token:

```text
B:
45 -> 46

C:
38 -> 39
```

The scheduler consumes:

```text
2 compute tokens
2 KV slots
```

for this iteration.

---

### Step 5

Another batched scheduling iteration gives:

```text
B = 47
C = 40

KV used:
87 / 88
```

Only:

```text
1 KV slot
```

remains.

Both B and C want to decode again.

Each requires one additional KV slot.

But only one slot exists.

This creates a resource conflict.

---

### Step 6

The scheduler processes requests in order.

It considers B first.

B needs:

```text
1 compute token
1 KV slot
```

The final free KV slot is available, so B is allowed to decode.

Its cache grows:

```text
47 -> 48
```

Now:

```text
free KV = 0
```

The scheduler then considers C.

C also wants one decode step.

Compute capacity is available.

But decoding C would require:

```text
C cache:
40 -> 41
```

There is no KV slot available.

So:

```text
KV-blocked decode:
['C']
```

This is an important behavior.

KV pressure is now affecting **already-running requests**, not just new admissions.

---

### B Finishes and Frees Memory

Fortunately, B reaches its generation limit during this step.

The engine immediately releases its entire cache:

```text
B = 48 KV slots
```

So after B finishes:

```text
KV allocations:

C = 40
```

and:

```text
48 slots become available again
```

That is why the engine does not permanently stall.

---

### Step 7

C can now continue:

```text
STEP 7

decoded:
['C']

finished:
['C']

running:
[]

waiting:
[]

KV allocations:
{}
```

C performs one final decode step:

```text
40 -> 41
```

and reaches its stopping condition.

Its cache is then freed.

Final KV state:

```text
used = 0
free = 88
```

Every request's KV memory has been reclaimed.

---

### Complete Execution Timeline

The full run looks like:

```text
STEP 1

A PREFILL
B PREFILL
C waits because compute budget is exhausted

KV:
A = 40
B = 43
```

Then:

```text
STEP 2

A DECODE
B DECODE
C waits because KV memory is insufficient

KV:
A = 41
B = 44
```

Then:

```text
STEP 3

A DECODE
B DECODE

A finishes
A's KV cache is freed

C PREFILL

KV:
B = 45
C = 38
```

Then:

```text
STEP 4

B DECODE
C DECODE
```

Then:

```text
STEP 5

B DECODE
C DECODE

KV almost full
```

Then:

```text
STEP 6

B DECODE

C blocked because no KV slot remains

B finishes
B's cache is freed
```

Finally:

```text
STEP 7

C DECODE

C finishes
C's cache is freed
```

The active request lifecycle was therefore:

```text
[]

↓

[A, B]

↓

[B, C]

↓

[C]

↓

[]
```

while KV ownership changed independently:

```text
{}

↓

{A: 40, B: 43}

↓

{A: 41, B: 44}

↓

{B: 45, C: 38}

↓

{B: 47, C: 40}

↓

{C: 40}

↓

{}
```

---

### Logical KV Manager vs Actual KV Cache

One important distinction in our implementation is that the `KVCacheManager` does not itself hold the K/V tensors.

Each request owns a real:

```python
DynamicCache
```

containing the actual tensors used by attention.

The manager instead tracks:

```text
how much KV memory each request is allowed to consume
```

So the relationship is:

```text
KVCacheManager
     ↓
resource accounting
     ↓
request allowed to run?
     ↓
yes
     ↓
real model forward
     ↓
real DynamicCache grows
```

Afterward we verify:

```text
manager allocation
==
actual cache sequence length
```

For example:

```text
A allocation = 41
A actual cache = 41

B allocation = 44
B actual cache = 44
```

This gives us a clean separation between:

```text
scheduling / resource policy
```

and:

```text
actual model KV tensors
```

That separation becomes increasingly important as the engine gets more sophisticated.

---

### Generated Outputs

Our requests produced:

```text
Request A:
The capital of
```

```text
Request B:
The largest planet in our solar
```

```text
Request C:
Machine learning is a
```

These outputs are intentionally short because the requests use very small `max_new_tokens` values.

The goal here is not response quality.

We deliberately want requests to finish at different times so that we can observe:

```text
KV allocation
KV growth
KV exhaustion
KV blocking
KV reclamation
new request admission
```

---

## Part 6 Conclusion

We now understand and have implemented:

```text
KV memory cost per token                 ✅

KV memory scaling with context length    ✅

concurrent-request KV usage              ✅

logical KV capacity                      ✅

per-request KV ownership                 ✅

KV allocation during prefill             ✅

KV growth during decode                  ✅

KV release when requests finish          ✅

compute-aware scheduling                 ✅

KV-capacity-aware scheduling             ✅

new requests blocked by KV pressure      ✅

running requests blocked by KV pressure  ✅

real DynamicCache integration            ✅

logical vs actual cache verification     ✅
```

Our scheduler no longer thinks only about compute.

It now reasons about two limited serving resources:

```text
COMPUTE
+
KV MEMORY
```

A request progresses only when both are available.

---

### The Remaining Problem

Our KV manager currently treats memory like one perfect pool of token slots:

```text
total = 88

used = 83

free = 5
```

This assumes that if five slots are free, those five slots can always be used wherever we want.

Real GPU memory is not that convenient.

Requests:

```text
arrive
grow
finish
free memory
```

at different times.

If every request expects one large contiguous KV allocation, memory can become fragmented.

Conceptually we might have:

```text
[AAAA][FREE][BBBBBB][FREE][CCC][FREE]
```

There may be plenty of free memory in total, but not necessarily one large contiguous region where a new request can grow.

We also do not want to repeatedly:

```text
move giant KV tensors
copy caches
reallocate contiguous buffers
compact memory
```

every time requests change.

This is the limitation that leads to the next major idea.

### Part 6 Complete

That's it for **Part 6: KV-Cache Management**.

We now know how to estimate KV memory, track ownership, grow and free cache capacity, and make scheduling decisions based on real KV pressure.

The next part is:

### Next: Part 7: Paged KV Cache Allocation and PagedAttention

Instead of treating each request's KV cache as one large contiguous region, we will break KV memory into fixed-size blocks and let a request own multiple blocks that do not need to be physically adjacent.

Conceptually:

```text
Request A logical KV:

tokens 0 ───────────────────────► 47
```

can map onto physical memory like:

```text
Block 7
Block 2
Block 19
```

rather than requiring:

```text
one giant contiguous allocation
```

That will let us understand:

```text
KV blocks
block tables
allocation
free lists
internal fragmentation
non-contiguous KV storage
```

and eventually the core idea behind **PagedAttention**.